# Suite SES — The session: candidates, copies, policies

A session is the caller of applications ([CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md)): `Transforms.Session(store, transforms,
history, procedures, source=..., target=..., frames=..., over=...)`. Its candidates are a filter fed by a frame
stream, as the user decided ("The transform candidates must come from a filter fed by a frame stream"): the frames a
mutation expression's execution gave back, in the store `over`, or by default only each transform's prototype mapped
over the source; `feed` extends the stream. Each frame is filtered once, for each transform whose prototype's symbols
it binds (and, where switched on, that its history has not taken), and expanded into pending candidates, once per
combination of the values of its finite parameters, in a canonical order. A session consumes its stream: termination
is relative to it ("In a streaming system termination conditions are relative"). Taking a candidate binds its
arguments (its frame, as the argument whose pattern is the prototype, carried by path into that argument's store; the
sync's frame of the same objects in the sync, the source's copy, which the session makes; the target's frame in the
caller's target store), applies its transform (`Transforms.apply`), and keeps the history certificate, which records
who decided each step, the caller (`take`) or a policy (`step_over`, `run`). A `Policy` of `Clause`s ranks pending
candidates and answers open parameters; it decides only where the caller lets it. Procedures never see the session.

The fixture is suite XF's shelf, with a note on each item, `shelf.Case` (a union of two options, `upper` and `lower`)
and `shelf.Size` (a union one of whose branches holds a value). Its patterns: `shelf.Unlabelled`, an item without a
label, declaring `case`; `shelf.Unnoted`, an item without a note, declaring `loud` (a bool) and `note` (a str);
`shelf.Unsized`, declaring `size` and `like` (for an object); `shelf.Blank`, an item without a label, declaring
none; `shelf.Any`, any item; `shelf.Whole`, the shelf; and `shelf.Kinds`, an object schema.

In [ ]:
from dataclasses import replace

from mbse.Schemas.Framework import Builders as B, Parametrics, Plain, Proxies
from mbse.Transforms import Transforms
from support import raises


def shelf(txn):
    """shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name, a label and a note;
    shelf.Case, a union of two options; and shelf.Size, a union one of whose branches holds an item."""
    S = txn.Schemas
    return txn.uid("ses.shelf/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Shelf": S.OfObject, "Item": S.OfObject, "Holds": S.OfRelation, "Upper": S.OfObject,
                  "Lower": S.OfObject, "Case": S.OfUnion, "Size": S.OfUnion})
        .spec([
            lambda _: _.Shelf.name("shelf.Shelf").ref(True).singleton("shelf.TheShelf").adjacencies({
                "items": lambda t: t.relation([_.Holds]).me("shelf")}),
            lambda _: _.Item.name("shelf.Item").ref(True).properties({
                "name": S.OfStr, "label": S.OfStr, "note": S.OfStr}).adjacencies([
                lambda t: t.name("shelf").relation([_.Holds]).me("item").owner(True)]),
            lambda _: _.Holds.name("shelf.Holds").links([lambda t: t.name("shelf"), lambda t: t.name("item")])
            .properties({"key": S.OfStr}).uniques([["shelf", "key"], ["item"]]),
            lambda _: _.Upper.name("shelf.Upper"),
            lambda _: _.Lower.name("shelf.Lower"),
            lambda _: _.Case.name("shelf.Case").branches([
                lambda b: b.name("upper").schema([_.Upper]), lambda b: b.name("lower").schema([_.Lower])]),
            lambda _: _.Size.name("shelf.Size").branches([
                lambda b: b.name("small").schema([_.Lower]), lambda b: b.name("custom").schema([_.Item])]),
        ])]))])


def patterns(txn):
    """shelf.Unlabelled, declaring case; shelf.Unnoted, declaring loud and note; shelf.Unsized, declaring size and
    like; shelf.Blank, an item without a label, declaring none; shelf.Any; shelf.Whole; and shelf.Kinds."""
    S, P = txn.Schemas, txn.Parametrics
    return txn.uid("ses.patterns/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Unlabelled": S.OfPattern, "Unnoted": S.OfPattern, "Unsized": S.OfPattern, "Blank": S.OfPattern,
                  "Any": S.OfPattern,
                  "Whole": S.OfPattern, "Kinds": S.OfPattern})
        .spec([
            lambda _: _.Unlabelled.name("shelf.Unlabelled").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.label(None)])
            .parameters([P.OfParameters(lambda ps: ps.parameters({
                "case": lambda d: d.parameter(lambda p: p.for_value([txn.shelf.Case]))}))]),
            lambda _: _.Unnoted.name("shelf.Unnoted").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.note(None)])
            .parameters([P.OfParameters(lambda ps: ps.parameters({
                "loud": lambda d: d.parameter(lambda p: p.for_value([S.OfBool])),
                "note": lambda d: d.parameter(lambda p: p.for_value([S.OfStr]))}))]),
            lambda _: _.Unsized.name("shelf.Unsized").symbols({"i": txn.shelf.Item})
            .parameters([P.OfParameters(lambda ps: ps.parameters({
                "size": lambda d: d.parameter(lambda p: p.for_value([txn.shelf.Size])),
                "like": lambda d: d.parameter(lambda p: p.for_object([txn.shelf.Item]))}))]),
            lambda _: _.Blank.name("shelf.Blank").symbols({"i": txn.shelf.Item}).spec([lambda f: f.i.label(None)]),
            lambda _: _.Any.name("shelf.Any").symbols({"i": txn.shelf.Item}),
            lambda _: _.Whole.name("shelf.Whole").symbols({"s": txn.shelf.Shelf}),
            lambda _: _.Kinds.name("shelf.Kinds").symbols({"o": S.OfObject}),
        ])]))])


def model(*names):
    """A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it."""
    store = B.Store()
    store.transaction(Parametrics.install)
    store.transaction(shelf)
    store.transaction(patterns)
    store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"s": t.shelf.Shelf})]))]))
    for name in names:
        with store.transaction():
            store.shelf.Shelf(store.singleton("shelf.TheShelf"), lambda s, n=name: s.items({
                n: lambda e: e.item(lambda i: i.name(n))}))
    return store


def items(store):
    """Each item on the shelf: its key, name, label and note."""
    with store.transaction():
        return [(e.key.get(), e.item.name.get(), e.item.label.get(), e.item.note.get())
                for e in store.singleton("shelf.TheShelf").items]


def key(store, reference):
    """The key an item is held under, read from its reference's path."""
    return Plain.ToPlain(store)(reference.as_schema(), reference)["path"]["next"]["entry"]["properties"]["key"]


def shown(store, candidates):
    """Each candidate: its transform's name, its item's key, its assignments and its open parameters."""
    return [(c.name, key(store, c.match["i"]), dict(c.assignments), c.open) for c in candidates]


def frames(store, pattern):
    """The frames of a pattern in a store, as its executor gives them back: a frame stream."""
    held = store.registered(pattern)
    return store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([held]))]))


def pick(store, candidates, name, item, **assignments):
    """The pending candidate of the transform `name` whose frame binds the item keyed `item`, with those
    assignments."""
    (found,) = [c for c in candidates if c.name == name and key(store, c.match["i"]) == item
                and dict(c.assignments) == assignments]
    return found


def decisions(store, history):
    """Who decided each step of a history, in order, and each step's transform."""
    with store.transaction():
        held = store.singleton("Transforms.TheHistories").histories.at(history).history
        return [(str(n.name), str(d.by)) for d in held.decisions for a in d.step.transform for n in a.transform.catalog]


certs = model()  # the certificates' store: the schemas and patterns, no data, and the transforms
certs.transaction(Transforms.install)
for define in (
    lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}, "falsified"),
    lambda t: Transforms.define(t, "Shout", "shout", t.shelf.Blank, {"sync": t.shelf.Blank}),
    lambda t: Transforms.define(t, "Note", "note", t.shelf.Unnoted, {"sync": t.shelf.Unnoted}),
    lambda t: Transforms.define(t, "Size", "note", t.shelf.Unsized, {"sync": t.shelf.Unsized}),
    lambda t: Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled, {"source": t.shelf.Unlabelled,
                                                                        "target": t.shelf.Whole}),
    lambda t: Transforms.define(t, "Stock", "stock", t.shelf.Any, {"source": t.shelf.Any, "target": t.shelf.Whole}),
    lambda t: Transforms.define(t, "Mirror", "mirror", t.shelf.Any, {"source": t.shelf.Any,
                                                                     "sync": t.shelf.Unlabelled}),
):
    certs.transaction(define)
procedures = Transforms.Procedures()


@procedures.register("label")
def label(source, sync, target, assignments):
    """Labels the sync's item with its name, in the case its parameter assignment gives."""
    name = str(sync.i.name)
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.label(name.upper() if assignments["case"] == "upper" else name))


@procedures.register("shout")
def shout(source, sync, target, assignments):
    """Labels the sync's item with its name and an exclamation mark."""
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.label(f"{sync.i.name}!"))


@procedures.register("note")
def note(source, sync, target, assignments):
    """Notes the sync's item, loudly or not."""
    text = assignments["note"].upper() if assignments["loud"] else assignments["note"]
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.note(text))


@procedures.register("copy")
def copy(source, sync, target, assignments):
    """Puts on the target's shelf an item named as the source's, in the case its parameter assignment gives."""
    name = str(source.i.name)
    name = name.upper() if assignments["case"] == "upper" else name
    Proxies.store_of(target.s).shelf.Shelf(target.s, lambda s: s.items({name: lambda e: e.item(lambda i: i.name(name))}))


@procedures.register("stock")
def stock(source, sync, target, assignments):
    """Puts on the target's shelf an item named as the source's."""
    copy(source, sync, target, {"case": "as is"})


@procedures.register("mirror")
def mirror(source, sync, target, assignments):
    """Labels the sync's item, the source's item copied, with the source item's name."""
    Proxies.store_of(sync.i).shelf.Item(sync.i, lambda i: i.label(f"mirror of {source.i.name}"))

<a id="SES-01"></a>
## SES-01 · Candidates: by default, each frame of a transform's prototype mapped over the source, once per combination of its finite parameters' values (a bool's; a union of options' names), any other parameter left open; in the canonical order

In [ ]:
source = model("bolt", "nut")
session = Transforms.Session(certs, ["Stock", "Label", "Note", "Size"], "candidates", procedures, source=source,
                             target=model())
assert shown(source, session.candidates()) == [
    # parameterized transforms first, then the transforms' order, the matches' order, each value's in its domain
    ("Label", "bolt", {"case": "upper"}, ()), ("Label", "bolt", {"case": "lower"}, ()),
    ("Label", "nut", {"case": "upper"}, ()), ("Label", "nut", {"case": "lower"}, ()),
    ("Note", "bolt", {"loud": False}, ("note",)), ("Note", "bolt", {"loud": True}, ("note",)),
    ("Note", "nut", {"loud": False}, ("note",)), ("Note", "nut", {"loud": True}, ("note",)),
    ("Size", "bolt", {}, ("size", "like")), ("Size", "nut", {}, ("size", "like")),  # neither domain finite
    ("Stock", "bolt", {}, ()), ("Stock", "nut", {}, ()),  # no parameters: last
]
first = session.candidates()[0]
assert first.transform is not None and first.score == 0.0 and sorted(first.match) == ["i"]
assert repr(first).startswith("Candidate(transform=") and "order" not in repr(first)
assert not session.done and session.steps == []
assert shown(source, session.candidates()) == shown(source, session.candidates())  # deterministic

<a id="SES-02"></a>
## SES-02 · Open parameters are assigned; a candidate with an open parameter, or one not pending, is refused; a candidate skipped is no longer pending

In [ ]:
data = model("bolt")
session = Transforms.Session(certs, ["Note", "Label"], "refusals", procedures, source=data)
loud = session.candidates()[1]
assert (loud.name, dict(loud.assignments), loud.open) == ("Note", {"loud": True}, ("note",))
answered = loud.answer(note="fragile")
assert (dict(answered.assignments), answered.open) == ({"loud": True, "note": "fragile"}, ())
assert loud.open == ("note",)  # a candidate is a value: answering gives another
with raises(TypeError, match="transform 'Note' has no open parameter 'loud'"):
    loud.answer(loud=False)
with raises(ValueError, match="transform 'Note' has open parameters ['note']: answer them first"):
    session.take(loud)
upper = session.candidates()[2]
assert (upper.name, dict(upper.assignments)) == ("Label", {"case": "upper"})
for unoffered in (replace(upper, assignments={"case": "title"}),  # no such value
                  replace(upper, assignments={}),  # its value not assigned
                  Transforms.Candidate(upper.transform, "Label", upper.match, {"case": "upper"})):  # from no stream
    with raises(ValueError, match="transform 'Label' is not a candidate for that match and those assignments: not pending, taken or skipped already"):
        session.take(unoffered)
session.skip(upper)  # the caller leaves it untaken
assert [dict(c.assignments) for c in session.candidates() if c.name == "Label"] == [{"case": "lower"}]
with raises(ValueError, match="not pending, taken or skipped already"):
    session.skip(upper)
session.take(answered)
assert items(session.sync) == [("bolt", "bolt", None, "FRAGILE")] and items(data) == [("bolt", "bolt", None, None)]
with raises(ValueError, match="not pending, taken or skipped already"):
    session.take(answered)

<a id="SES-03"></a>
## SES-03 · Taking a candidate: the session binds its arguments and applies its transform, by the caller's decision; the frame's other candidates for that transform leave, and nothing is queried again

In [ ]:
source, target = model("bolt", "nut"), model()
session = Transforms.Session(certs, ["Label", "Shout", "Copy"], "take", procedures, source=source, target=target)
bolt_lower = pick(source, session.candidates(), "Label", "bolt", case="lower")
certificate = session.take(bolt_lower)  # the source's frame of Label's prototype, carried into the sync
assert session.steps == [certificate]
assert items(session.sync) == [("bolt", "bolt", "bolt", None), ("nut", "nut", None, None)]  # in the sync, in place
assert shown(source, session.candidates()) == [
    ("Label", "nut", {"case": "upper"}, ()), ("Label", "nut", {"case": "lower"}, ()),  # bolt's: one decision, taken
    ("Copy", "bolt", {"case": "upper"}, ()), ("Copy", "bolt", {"case": "lower"}, ()),
    ("Copy", "nut", {"case": "upper"}, ()), ("Copy", "nut", {"case": "lower"}, ()),
    ("Shout", "bolt", {}, ()), ("Shout", "nut", {}, ()),  # pending still: the stream is consumed, not queried again
]
session.skip(pick(source, session.candidates(), "Shout", "bolt"))  # the competing step, left untaken by the caller
session.take(pick(source, session.candidates(), "Shout", "nut"))
session.take(pick(source, session.candidates(), "Copy", "bolt", case="upper"))  # the target's one frame of shelf.Whole
assert items(session.sync) == [("bolt", "bolt", "bolt", None), ("nut", "nut", "nut!", None)]
assert items(target) == [("BOLT", "BOLT", None, None)]
assert decisions(certs, "take") == [("Label", "caller"), ("Shout", "caller"), ("Copy", "caller")]

<a id="SES-04"></a>
## SES-04 · A session consumes its stream: a frame is offered once, and again only when fed again, unless the session excludes what its history took; it is done when the stream is drained and nothing is pending

In [ ]:
source, target = model("bolt", "nut"), model()
session = Transforms.Session(certs, ["Copy", "Stock"], "again", procedures, source=source, target=target)
assert len(session.candidates()) == 6 and session.drained  # a list: filtered and expanded when fed
session.take(session.candidates()[0])  # Copy bolt upper
while not session.done:  # termination, relative to the input: the frames fed, each decided once
    session.take(session.candidates()[0])
assert len(session.steps) == 4 and session.candidates() == []
assert items(source) == [("bolt", "bolt", None, None), ("nut", "nut", None, None)]  # still matches there: not asked
session.feed(frames(source, "shelf.Unlabelled"))  # the same frames, fed again: offered again
assert shown(source, session.candidates()) == [
    ("Copy", "bolt", {"case": "upper"}, ()), ("Copy", "bolt", {"case": "lower"}, ()),
    ("Copy", "nut", {"case": "upper"}, ()), ("Copy", "nut", {"case": "lower"}, ()),
    ("Stock", "bolt", {}, ()), ("Stock", "nut", {}, ()),  # a frame binding Stock's prototype's symbols too
]

session = Transforms.Session(certs, ["Copy", "Stock"], "excluded", procedures, source=source, target=model(),
                             exclude_taken=True)  # the filter on: what the history took is left out when fed again
session.take(session.candidates()[0])  # Copy bolt upper
session.feed(frames(source, "shelf.Unlabelled"))
assert shown(source, session.candidates()) == [
    # pending from the first feed, and fed again, in the canonical order: Copy's of bolt left out, taken
    ("Copy", "nut", {"case": "upper"}, ()), ("Copy", "nut", {"case": "upper"}, ()),
    ("Copy", "nut", {"case": "lower"}, ()), ("Copy", "nut", {"case": "lower"}, ()),
    ("Stock", "bolt", {}, ()), ("Stock", "bolt", {}, ()), ("Stock", "nut", {}, ()), ("Stock", "nut", {}, ()),
]
with certs.transaction():  # what tells: the step's transform, and its sub-certificate on the source's frame
    (step,) = (e.step for e in certs.singleton("Transforms.TheHistories").histories.at("excluded").history.steps)
    assert [str(n.name) for a in step.transform for n in a.transform.catalog] == ["Copy"]
    assert [str(e.argument) for e in step.frames] == ["source", "target"]

<a id="SES-05"></a>
## SES-05 · Copies are the session's: the sync a copy of the source it makes from a snapshot, the source never disturbed, the target the caller's; a frame is carried into the copy by its objects' paths

In [ ]:
source = model("bolt", "nut")
session = Transforms.Session(certs, ["Mirror", "Label"], "copies", procedures, source=source)
assert session.sync is not source and session.target is None
assert items(session.sync) == items(source)  # made from the source's snapshot: every object at the same path
assert Transforms.Session(certs, ["Copy"], "none", procedures, source=source, target=model()).sync is None  # none taken
session.take(pick(source, session.candidates(), "Mirror", "nut"))  # the sync's frame: nut, as copied
assert items(session.sync) == [("bolt", "bolt", None, None), ("nut", "nut", "mirror of nut", None)]
session.take(pick(source, session.candidates(), "Label", "bolt", case="upper"))  # the source's frame, carried
assert items(session.sync) == [("bolt", "bolt", "BOLT", None), ("nut", "nut", "mirror of nut", None)]
assert items(source) == [("bolt", "bolt", None, None), ("nut", "nut", None, None)]  # undisturbed
with raises(ValueError, match="transform 'Mirror': the sync's store holds 0 frames of 'shelf.Unlabelled' binding ['i'] as the match does: a session binds the sync frame to its one frame"):
    session.take(pick(source, session.candidates(), "Mirror", "bolt"))  # bolt is labelled in the sync
assert len(session.steps) == 2

seen = []  # a frame binding a schema object: the sync's binds the copy's, by its name
procedures.register("kinds", lambda source, sync, target, assignments: seen.append(
    (source.o.name, sync.o.name, sync.o is source.o)))
certs.transaction(lambda t: Transforms.define(t, "Kinds", "kinds", t.shelf.Kinds, {"source": t.shelf.Kinds,
                                                                                   "sync": t.shelf.Kinds}))
session = Transforms.Session(certs, ["Kinds"], "kinds", procedures, source=source)
(item_kind,) = [c for c in session.candidates() if c.match["o"].name == "shelf.Item"]
session.take(item_kind)
assert seen == [("shelf.Item", "shelf.Item", False)]
assert [c for c in session.candidates() if c.match["o"].name == "shelf.Item"] == []

<a id="SES-06"></a>
## SES-06 · Policies rank: a clause weighs the candidates of the transforms it names by reference that agree with its assignments, an open parameter agreeing; a score is the sum of the weights, ties in the canonical order

In [ ]:
Label, Copy, Note = (Transforms.reference(certs, name) for name in ("Label", "Copy", "Note"))
policy = Transforms.Policy(
    Transforms.Clause(Label, {"case": "lower"}, 2.0),
    Transforms.Clause([Label, Copy], weight=0.5),  # one clause, several transforms: those sharing a reason
    Transforms.Clause(Note, {"note": "fragile"}),  # agrees with an open parameter, and answers it
    Transforms.Clause(Note, {"loud": 1}),  # 1 is not True: no candidate agrees
    Transforms.Clause(Copy, {"size": "small"}),  # Copy has no size: it weighs none of its candidates
)
data = model("bolt")
session = Transforms.Session(certs, ["Copy", "Label", "Note", "Stock"], "policy", procedures, source=data,
                             target=model())
ranked = session.candidates(policy)
assert [(c.name, dict(c.assignments), c.score) for c in ranked] == [
    ("Label", {"case": "lower"}, 2.5),
    ("Note", {"loud": False}, 1.0), ("Note", {"loud": True}, 1.0),
    ("Copy", {"case": "upper"}, 0.5), ("Copy", {"case": "lower"}, 0.5), ("Label", {"case": "upper"}, 0.5),
    ("Stock", {}, 0.0),  # no clause weighs it: not the policy's to take
]
assert [c.score for c in session.candidates()] == [0.0] * 7  # without a policy, unscored, in the canonical order
assert policy.answers(ranked[1]) == {"note": "fragile"} and policy.answers(ranked[0]) == {}
elsewhere = model()  # a reference is the catalog's: another store's transform named Label is another transform
elsewhere.transaction(Transforms.install)
elsewhere.transaction(lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}))
assert Transforms.Policy(Transforms.Clause(Transforms.reference(elsewhere, "Label"))).score(ranked[0]) == 0
with raises(TypeError, match="a clause names a transform by its reference, Transforms.reference(store, name), or a sequence of them"):
    Transforms.Clause("Label")
with raises(LookupError, match="no transform named 'Nothing' is defined in the store"):
    Transforms.reference(certs, "Nothing")

<a id="SES-07"></a>
## SES-07 · A policy decides where the caller lets it: `step_over` takes its first candidate, answering open parameters, and `run` steps until it cannot; the history records who decided each step

In [ ]:
Label, Note, Copy = (Transforms.reference(certs, name) for name in ("Label", "Note", "Copy"))
policy = Transforms.Policy(Transforms.Clause(Label, {"case": "upper"}, 2.0), Transforms.Clause(Note, {"note": "ok"}))
source, target = model("bolt", "nut"), model()
session = Transforms.Session(certs, ["Label", "Note", "Copy"], "steps", procedures, source=source, target=target)
assert session.step_over() is False and session.run() == 0  # no policy: every choice the caller's
assert session.step_over(policy) is True and items(session.sync)[0] == ("bolt", "bolt", "BOLT", None)
assert session.run(policy) == 3  # Label nut; then Note bolt and nut, loud False first, their note answered
assert items(session.sync) == [("bolt", "bolt", "BOLT", "ok"), ("nut", "nut", "NUT", "ok")]
assert not session.done and session.step_over(policy) is False  # only Copy is left, which no clause weighs
session.take(session.candidates()[1])  # the caller decides: Copy bolt lower
assert decisions(certs, "steps") == [("Label", "policy"), ("Label", "policy"), ("Note", "policy"),
                                     ("Note", "policy"), ("Copy", "caller")]

data = model("bolt")  # a policy that cannot answer an open parameter takes nothing
session = Transforms.Session(certs, ["Note"], "unanswered", procedures, source=data)
assert session.run(Transforms.Policy(Transforms.Clause(Note, {"loud": True}))) == 0 and len(session.candidates()) == 2

session = Transforms.Session(certs, ["Label"], "batch", procedures, source=model("bolt", "nut"))
assert session.run(Transforms.Policy(Transforms.Clause(Label))) == 2  # a policy that decides everything: a batch
assert session.done and session.candidates() == [] and session.step_over(policy) is False

<a id="SES-08"></a>
## SES-08 · What a session refuses: a history that holds steps, a transform with no prototype argument or whose target it is not given, and an argument it cannot bind to one frame

In [ ]:
source = model("bolt")
certs.transaction(lambda t: Transforms.define(t, "Orphan", "copy", t.shelf.Unlabelled, {"target": t.shelf.Whole}))
for transforms, history, target, error, message in [
    (["Label"], "take", None, ValueError, "history 'take' holds 3 steps already: a session keeps a history of its own, begun empty"),
    (["Copy"], "new", None, ValueError, "transform 'Copy' takes a target: the session is given none"),
    (["Orphan"], "new", model(), ValueError, "transform 'Orphan' takes neither a source nor a sync: its prototype is the pattern of one"),
    (["Nothing"], "new", None, LookupError, "no transform named 'Nothing' is defined in the store"),
]:
    with raises(error, match=message):
        Transforms.Session(certs, transforms, history, procedures, source=source, target=target)
certs.transaction(lambda t: Transforms.define(t, "Restock", "stock", t.shelf.Any, {"source": t.shelf.Any,
                                                                                    "target": t.shelf.Any}))
session = Transforms.Session(certs, ["Restock"], "restock", procedures, source=source, target=model("a", "b"))
with raises(ValueError, match="transform 'Restock': the target's store holds 2 frames of 'shelf.Any': a session binds the target frame to its one frame"):
    session.take(session.candidates()[0])
assert session.steps == [] and items(source) == [("bolt", "bolt", None, None)]

<a id="SES-09"></a>
## SES-09 · The caller feeds the stream: a session offers exactly the frames fed, in the store they are in, that bind a transform's prototype's symbols, expanded; an iterable is pulled a frame at a time, and frames fed later extend the stream

In [ ]:
source = model("bolt", "nut", "washer")
subset = [frame for frame in frames(source, "shelf.Unlabelled") if key(source, frame["i"]) != "nut"]  # filtered
session = Transforms.Session(certs, ["Label"], "fed", procedures, source=source, frames=subset)
assert shown(source, session.candidates()) == [
    ("Label", "bolt", {"case": "upper"}, ()), ("Label", "bolt", {"case": "lower"}, ()),
    ("Label", "washer", {"case": "upper"}, ()), ("Label", "washer", {"case": "lower"}, ()),
]
session.feed(frames(source, "shelf.Whole"))  # frames that do not bind its prototype's symbols: out of its scope
assert len(session.candidates()) == 4

session = Transforms.Session(certs, ["Label", "Shout"], "over", procedures, source=source, frames=[])
assert session.done
sync = session.sync
session.feed(frames(sync, "shelf.Blank"), over=sync)  # frames over the sync, as fed: bound as they are
session.take(pick(sync, session.candidates(), "Label", "nut", case="upper"))
session.feed(frames(sync, "shelf.Blank"), over=sync)  # the sync queried again, by the caller: nut matches no more
assert [(c.name, key(sync, c.match["i"])) for c in session.candidates() if c.name == "Shout"] == [
    ("Shout", "bolt"), ("Shout", "bolt"), ("Shout", "nut"), ("Shout", "washer"), ("Shout", "washer")]
assert items(sync)[1] == ("nut", "nut", "NUT", None) and items(source)[1] == ("nut", "nut", None, None)

Label = Transforms.reference(certs, "Label")
session = Transforms.Session(certs, ["Label", "Stock"], "pulled", procedures, source=source, target=model(),
                             frames=(frame for frame in frames(source, "shelf.Unlabelled")))  # an iterable
assert not session.drained and shown(source, session.candidates()) == [  # one frame pulled, as needed
    ("Label", "bolt", {"case": "upper"}, ()), ("Label", "bolt", {"case": "lower"}, ()), ("Stock", "bolt", {}, ())]
assert session.run(Transforms.Policy(Transforms.Clause(Label, {"case": "lower"}))) == 3  # pulled as it runs
assert session.drained and not session.done and items(session.sync) == [
    ("bolt", "bolt", "bolt", None), ("nut", "nut", "nut", None), ("washer", "washer", "washer", None)]
assert [c.name for c in session.candidates()] == ["Stock"] * 3  # no clause weighs them: the caller's
session.feed(iter([]))  # an iterable fed later extends the stream
assert not session.drained
for candidate in session.candidates():
    session.skip(candidate)
assert session.done and session.drained